### Using OpenAI (GPT-5.4) for data annotation

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from openai import OpenAI 
import json
from util.preprocessing import parse_iob2_file

#### Toy experiment with annotation for one sentence

In [8]:
one_shot = """
1	350	O
2	,	O
3	Wellesley	B-LOC
4	,	O
5	Massachusetts	B-LOC
6	02481	O
7	doing	O
8	business	O
9	as	O
10	"	O
11	Silicon	B-LOC
12	Valley	I-LOC
13	East	I-LOC
14	"	O
15	and	O
16	AKAMAI	B-ORG
17	TECHNOLOGIES	I-ORG
18	,	O
19	INC	O
20	.	O
21	("	O
22	Borrower	B-PER
23	"),	O
"""

one_shot = one_shot.strip().split('\n')

example_input_lines = []
example_output_lines = []

for line in one_shot:
    _, token, label = line.strip().split('\t')
    example_input_lines.append(token)
    example_output_lines.append(f"{token}\t{label}")

example_input = '\n'.join(example_input_lines)
example_output = '\n'.join(example_output_lines)


test_case = """1	1	O
2	.	O
3	3	O
4	Borrower	B-PER
5	agrees	O
6	to	O
7	accept	O
8	the	O
9	aforementioned	O
10	Loan	O
11	provided	O
12	by	O
13	Lender	B-PER
14	,	O
15	and	O
16	hereby	O
17	agrees	O
18	and	O
19	warrants	O
20	that	O
21	the	O
22	Loan	O
23	shall	O
24	be	O
25	used	O
26	solely	O
27	to	O
28	fund	O
29	its	O
30	contribution	O
31	to	O
32	the	O
33	registered	O
34	capital	O
35	of	O
36	[	O
37	Lenovo	B-ORG
38	Security	I-ORG
39	Technology	I-ORG
40	Ltd	I-ORG
41	.	O
42	(	O
43	hereinafter	O
44	the	O
45	“	O
46	Borrower	B-PER
47	Company	O
48	”,	O
49	a	O
50	domestic	O
51	-	O
52	funded	O
53	limited	O
54	liability	O
55	company	O
56	in	O
57	China	B-LOC
58	with	O
59	registered	O
60	capital	O
61	of	O
62	Renminbi	O
63	Twenty	O
64	Four	O
65	Million	O
66	(	O
67	RMB24	O
68	,	O
69	000	O
70	,	O
71	000	O
72	.	O
73	00	O
74	)).	O"""


test_case = test_case.strip().split('\n')

test_input_lines = []
test_output_lines = []

for line in test_case:
    _, token, label = line.strip().split('\t')
    test_input_lines.append(token)
    test_output_lines.append(f"{token}\t{label}")

test_input = '\n'.join(test_input_lines)
test_output = '\n'.join(test_output_lines)

print(example_input)
print(example_output)
print(test_input)
print(test_output)

350
,
Wellesley
,
Massachusetts
02481
doing
business
as
"
Silicon
Valley
East
"
and
AKAMAI
TECHNOLOGIES
,
INC
.
("
Borrower
"),
350	O
,	O
Wellesley	B-LOC
,	O
Massachusetts	B-LOC
02481	O
doing	O
business	O
as	O
"	O
Silicon	B-LOC
Valley	I-LOC
East	I-LOC
"	O
and	O
AKAMAI	B-ORG
TECHNOLOGIES	I-ORG
,	O
INC	O
.	O
("	O
Borrower	B-PER
"),	O
1
.
3
Borrower
agrees
to
accept
the
aforementioned
Loan
provided
by
Lender
,
and
hereby
agrees
and
warrants
that
the
Loan
shall
be
used
solely
to
fund
its
contribution
to
the
registered
capital
of
[
Lenovo
Security
Technology
Ltd
.
(
hereinafter
the
“
Borrower
Company
”,
a
domestic
-
funded
limited
liability
company
in
China
with
registered
capital
of
Renminbi
Twenty
Four
Million
(
RMB24
,
000
,
000
.
00
)).
1	O
.	O
3	O
Borrower	B-PER
agrees	O
to	O
accept	O
the	O
aforementioned	O
Loan	O
provided	O
by	O
Lender	B-PER
,	O
and	O
hereby	O
agrees	O
and	O
warrants	O
that	O
the	O
Loan	O
shall	O
be	O
used	O
solely	O
to	O
fund	O
its	O
contribution	O
to	O
the	O
registe

In [9]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4",
    input=(

    "You are a strict NER tagger.\n\n"

    "Task:\n"
    "Assign a BIO tag to EACH token.\n\n"

    "Allowed labels:\n"
    "B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, O\n\n"

    "Rules:\n"
    "- Annotate legal party roles \"Lender\" and \"Borrower\" as B-PER\n"
    "- EXACTLY one label per token\n"
    "- SAME number of output lines as input tokens\n"
    "- SAME token order as input\n"
    "- Do not modify tokens\n"
    "- Output format must be: token<TAB>label\n"
    "- One token-label pair per line\n"
    "- No explanations\n\n"

    "Example input:\n"
    f"{example_input}\n\n"

    "Example output:\n"
    f"{example_output}\n\n"

    "Now annotate this input:\n"
    f"{test_input}"
    )
)

openai_response = response.output[0].content[0].text
print(openai_response)

1	O
.	O
3	O
Borrower	B-PER
agrees	O
to	O
accept	O
the	O
aforementioned	O
Loan	O
provided	O
by	O
Lender	B-PER
,	O
and	O
hereby	O
agrees	O
and	O
warrants	O
that	O
the	O
Loan	O
shall	O
be	O
used	O
solely	O
to	O
fund	O
its	O
contribution	O
to	O
the	O
registered	O
capital	O
of	O
[	O
Lenovo	B-ORG
Security	I-ORG
Technology	I-ORG
Ltd	I-ORG
.	I-ORG
(	O
hereinafter	O
the	O
“	O
Borrower	O
Company	O
”,	O
a	O
domestic	O
-	O
funded	O
limited	O
liability	O
company	O
in	O
China	B-LOC
with	O
registered	O
capital	O
of	O
Renminbi	O
Twenty	O
Four	O
Million	O
(	O
RMB24	O
,	O
000	O
,	O
000	O
.	O
00	O
)).	O


In [10]:
openai_labels = openai_response.strip().split('\n')
true_labels = test_output.strip().split('\n')

assert len(openai_labels) == len(true_labels), "Number of labels does not match number of tokens."

print(f"TOKEN\tTOKEN_MATCH\tPREDICTED_LABEL\tTRUE_LABEL\tLABEL_MATCH")
for pred, true in zip(openai_labels, true_labels):
    pred_token, pred_label = pred.split('\t')
    true_token, true_label = true.split('\t')
    
    token_match = pred_token == true_token

    label_match = "✓" if pred_label == true_label else "✗"
    print(f"{pred_token}\t{token_match}\t{pred_label}\t{true_label}\t{label_match}")

TOKEN	TOKEN_MATCH	PREDICTED_LABEL	TRUE_LABEL	LABEL_MATCH
1	True	O	O	✓
.	True	O	O	✓
3	True	O	O	✓
Borrower	True	B-PER	B-PER	✓
agrees	True	O	O	✓
to	True	O	O	✓
accept	True	O	O	✓
the	True	O	O	✓
aforementioned	True	O	O	✓
Loan	True	O	O	✓
provided	True	O	O	✓
by	True	O	O	✓
Lender	True	B-PER	B-PER	✓
,	True	O	O	✓
and	True	O	O	✓
hereby	True	O	O	✓
agrees	True	O	O	✓
and	True	O	O	✓
warrants	True	O	O	✓
that	True	O	O	✓
the	True	O	O	✓
Loan	True	O	O	✓
shall	True	O	O	✓
be	True	O	O	✓
used	True	O	O	✓
solely	True	O	O	✓
to	True	O	O	✓
fund	True	O	O	✓
its	True	O	O	✓
contribution	True	O	O	✓
to	True	O	O	✓
the	True	O	O	✓
registered	True	O	O	✓
capital	True	O	O	✓
of	True	O	O	✓
[	True	O	O	✓
Lenovo	True	B-ORG	B-ORG	✓
Security	True	I-ORG	I-ORG	✓
Technology	True	I-ORG	I-ORG	✓
Ltd	True	I-ORG	I-ORG	✓
.	True	I-ORG	O	✗
(	True	O	O	✓
hereinafter	True	O	O	✓
the	True	O	O	✓
“	True	O	O	✓
Borrower	True	O	B-PER	✗
Company	True	O	O	✓
”,	True	O	O	✓
a	True	O	O	✓
domestic	True	O	O	✓
-	True	O	O	✓
funded	True	O	O	✓
limited	True	O	O	✓
liab

#### Handling nested structure of sentences and tokens to annotate multiple sentences in one prompt

In [ ]:
parsed = parse_iob2_file("FIN5_validation.txt")
from annotate_data import build_chunks

contract_5_sentences = parsed[0]
contract_5_labels = parsed[1]

chunks = build_chunks(contract_5_sentences, contract_5_labels, max_chunk_size=250)

with open("chunks.json", "w") as f:
    json.dump(chunks, f, indent=4, default=lambda x: float(x))

c:\Users\marib\anaconda3\envs\nlp-project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
contract_5_sentences = parsed[0]
contract_5_labels = parsed[1]

chunks

[{'size': 215,
  'tokens': ['Loan',
   'Agreement',
   'This',
   'Loan',
   'Agreement',
   '(',
   'this',
   '“',
   'Agreement',
   '”)',
   'is',
   'made',
   'and',
   'entered',
   'into',
   'by',
   'and',
   'between',
   'the',
   'parties',
   'listed',
   'below',
   'as',
   'of',
   'the',
   '19th',
   'day',
   'of',
   'October',
   ',',
   '2004',
   'in',
   'Beijing',
   ':',
   '(',
   '1',
   ')',
   'Lenovo',
   '-',
   'AsiaInfo',
   'Technologies',
   ',',
   'Inc',
   '.',
   '(“',
   'Lender',
   '”),',
   'a',
   'limited',
   'company',
   'duly',
   'organized',
   'and',
   'existing',
   'under',
   'the',
   'laws',
   'of',
   'the',
   'People',
   '’',
   's',
   'Republic',
   'of',
   'China',
   '(“',
   'PRC',
   '”',
   'or',
   '“',
   'China',
   '”)',
   'with',
   'its',
   'address',
   'at',
   '3',
   '/',
   'F',
   'Zhongdian',
   'Information',
   'Tower',
   ',',
   'No',
   '.',
   '6',
   'Zhongguancun',
   'South',
   'Street',
 